# A06_reasons: why they moved (keywords v0)

| Step | Purpose |
|---|---|
| 6.1 | Codebook v0 in `docs/CODEBOOK.md` (code, meaning, keywords, example) |
| 6.2 | Reason statements: the text after "reasons for removing" in `reason_pages` |
| 6.3 | Code each statement (a statement can carry several codes) |
| 6.4 | Declaration hits: reason keywords within 150 characters of "removed / moved / emigrated" |
| 6.5 | Long table + save `interim/reasons` |

Regex runs in **DuckDB** (RE2, linear time). A pandas regex with a leading `.{0,150}` backtracks and took >10 min.

In [1]:
import sys
from pathlib import Path
PIPELINE_DIR = Path.cwd().parent
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
import config
from rx import io, text, places, remote
import pandas as pd

## 6.1 Codebook
Keywords use `\b` word boundaries: without them `son` matches "rea**son**" and every statement came out "family".

In [2]:
con = remote.connect()
rp = io.read_stage("interim", "reason_pages")
d = io.read_stage("interim", "decl_text", columns=["NAID", "pageNumber", "text"])
con.register("rp", rp); con.register("d", d)

REASON_RX = r"reasons? (?:for|of) (?:his |her |my )?(?:removing|removal|having removed|moving|leaving)|cause of (?:his |her )?remov"
CODES = {
    "family":        r"\b(?:child|children|sons?|daughters?|sons?.in.law|grand\w*|relatives?|relations?|kindred|friends|brothers?|sisters?|family|connexions?)\b",
    "age_health":    r"\b(?:aged|old age|old|infirm\w*|feeble|blind|palsied|sick\w*|health|helpless|decrepit)\b",
    "poverty":       r"\b(?:poor|poverty|support|maintain|destitute|means|indigent|livelihood|dependent)\b",
    "land_work":     r"\b(?:land|lands|farm|cheap|employment|business|work|trade|settle|prospects)\b",
    "spouse":        r"\b(?:husband|wife|married|marriage)\b",
    "pension_admin": r"\b(?:agent|agency|certificate|draw (?:his|her|my) pension)\b",
    "climate":       r"\b(?:climate|healthier|sickly)\b",
}
len(rp), rp.NAID.nunique()

(2493, 2079)

## 6.2 Reason statements
Pull up to 300 characters after each trigger, then strip the trigger so only what follows is coded.

In [3]:
s = con.sql(f"""select NAID, pageNumber,
                       unnest(regexp_extract_all(text, '(?:{REASON_RX}).{{0,300}}')) as snippet
                from rp""").df()
s["stmt"] = s.snippet.str.replace(rf"^(?:{REASON_RX})", "", regex=True)
len(s), s.NAID.nunique()

(2506, 2079)

## 6.3 Code each statement

In [4]:
for c, rx in CODES.items():
    s[c] = s.stmt.str.contains(rx)
print("no code:", (~s[list(CODES)].any(axis=1)).sum())
a = s.melt(id_vars=["NAID", "pageNumber", "snippet"], value_vars=list(CODES), var_name="reason_code")
a = a[a.value].drop(columns="value").assign(source="reason_statement")
a.groupby("reason_code").NAID.nunique().sort_values(ascending=False)

no code: 572


reason_code
family           1537
age_health        373
spouse            210
poverty           205
land_work         169
pension_admin      62
climate            21
Name: NAID, dtype: int64

## 6.4 Declaration hits near a move
Across the declaration text most keyword hits are legal formula (the 1818 Act *required* poverty) or service stories.
A hit counts only when "removed / moved / emigrated" sits in the same 300-character snippet.

In [5]:
DECL = {
    "land":       r"bounty land|locate[d]? (?:his|my|the) (?:land|warrant)|purchase[d]? (?:a )?(?:farm|land|tract)|new (?:country|settlement)|wild lands?|took up land",
    "family":     r"(?:live|reside|resided|living) with (?:his|her|my) (?:son|daughter|children|son-in-law|friends|relations)|(?:his|her|my) children (?:had )?(?:removed|moved)",
    "poverty":    r"poverty|indigent|destitute|reduced circumstances|unable to (?:support|maintain)|in need of (?:the )?assistance|no property",
    "health":     r"infirmit|ill health|sickness|disease|blind|lame|rheumati|deaf",
    "war_damage": r"(?:burnt|burned|plundered|destroyed|taken) by the (?:enemy|british|tories|indians)",
    "conflict":   r"\bindians? (?:came|attacked|killed|massacre)|frontier|fort (?:to|for) (?:protection|safety)|driven (?:off|away|from)",
}
q = " union all ".join(
    f"select NAID, pageNumber, '{c}' as reason_code, unnest(regexp_extract_all(text, '.{{0,150}}(?:{rx}).{{0,150}}')) as snippet from d"
    for c, rx in DECL.items())
b = con.sql(q).df()
print("keyword hits anywhere:", len(b))
b = b[b.snippet.str.contains(r"removed|moved|emigrated")].assign(source="declaration_near_move")
b.groupby("reason_code").NAID.nunique()

keyword hits anywhere: 80974


reason_code
conflict      119
family        210
health        161
land           40
poverty        55
war_damage     44
Name: NAID, dtype: int64

## 6.5 Long table + save

In [6]:
reasons = (pd.concat([a, b], ignore_index=True)
             .sort_values("pageNumber").drop_duplicates(["NAID", "reason_code", "source"]))
reasons["method"] = "keyword v0"
assert reasons.NAID.notna().all()
assert not reasons.duplicated(["NAID", "reason_code", "source"]).any()
io.write_stage(reasons, "interim", "reasons")
print(len(reasons), "rows in", reasons.NAID.nunique(), "files")
reasons.groupby(["source", "reason_code"]).NAID.nunique()

3206 rows in 2078 files


source                 reason_code  
declaration_near_move  conflict          119
                       family            210
                       health            161
                       land               40
                       poverty            55
                       war_damage         44
reason_statement       age_health        373
                       climate            21
                       family           1537
                       land_work         169
                       pension_admin      62
                       poverty           205
                       spouse            210
Name: NAID, dtype: int64

## Notes
- Reason statements are pension **transfers** sworn in the 1820s–1850s, when the applicant was old. They explain **late-life** moves, mostly to follow children. They don't explain the first post-war move.
- A07 precision: the statement codes hold up (family, age_health, land_work, climate ≥ 80%). The declaration-near-move codes (`health`, `war_damage`, `conflict`) are mostly service narrative, so leave them out of the viz.